In [1]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [2]:
from pathlib import Path
import sys
import pandas as pd
from icdn import PanelSchema
from icdn.data.splits import TemporalSplitter

ROOT = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent
sys.path.insert(0, str(ROOT))

from src.benchmarks.features import ICDNFeaturePipeline
from src.benchmarks.pairs import PairDatasetBuilder
from src.benchmarks.pairwise_ridge import PairwiseRidge

DATASETS = {
    "dunnhumby": {
        "path": ROOT / "data" / "dunnhumby" / "panel" / "dunnhumby_icdn_panel.parquet",
        "schema": PanelSchema(category="category", brand="brand", style="style"),
    },
    "walmart": {
        "path": ROOT / "data" / "M5-walmart" / "panel" / "m5_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
    "one_c": {
        "path": ROOT / "data" / "predict-future-sales-1c" / "panel" / "1c_icdn_panel.parquet",
        "schema": PanelSchema(category="category"),
    },
}

SHORT = ["promo", "sin_52", "cos_52"]

def run_one(name, spec):
    panel = pd.read_parquet(spec["path"])
    panel = panel[(panel["price"] > 0) & (panel["units"] > 0)].copy()
    print(f"\n=== {name} === {panel.shape}  "
          f"products={panel['product_code'].nunique()}  "
          f"stores={panel['store_code'].nunique()}")

    splitter = TemporalSplitter(period_col="week_id")
    train_raw, val_raw = splitter.single_split(panel, train_frac=0.8)

    feats = ICDNFeaturePipeline(schema=spec["schema"])
    train = feats.fit(train_raw).transform(train_raw)
    val = feats.transform_val(val_raw)
    print("controls:", len(SHORT), SHORT)

    builder = PairDatasetBuilder(SHORT)
    train_pairs = builder.build(train)
    val_pairs = builder.build(val)
    print("pairs train/val:", train_pairs.shape, val_pairs.shape)

    ridge = PairwiseRidge(SHORT)
    own = ridge.run_own(train, val)
    cross = ridge.run_cross(train_pairs, val_pairs)

    print(f"own: {len(own)} products estimated")
    print(f"cross: {len(cross)} pairs estimated")
    if len(own):
        print("own  mean/min/max", own.own_elasticity.mean(), own.own_elasticity.min(), own.own_elasticity.max())
    if len(cross):
        print("cross mean/min/max", cross.cross_elasticity.mean(), cross.cross_elasticity.min(), cross.cross_elasticity.max())
    return own, cross

for name, spec in DATASETS.items():
    run_one(name, spec)


=== dunnhumby === (3216, 9)  products=10  stores=20
controls: 3 ['promo', 'sin_52', 'cos_52']
pairs train/val: (5104, 10) (1460, 10)
own: 10 products estimated
cross: 26 pairs estimated
own  mean/min/max -0.8823073125487422 -1.326729677187593 -0.5565149913781721
cross mean/min/max -0.195815405953821 -3.2309599272049967 3.682979637763175

=== walmart === (47904, 7)  products=20  stores=10
controls: 3 ['promo', 'sin_52', 'cos_52']
pairs train/val: (681976, 10) (123876, 10)
own: 20 products estimated
cross: 380 pairs estimated
own  mean/min/max -0.656178639802152 -2.610007944848888 0.8758740190507618
cross mean/min/max -0.09519863412118228 -3.237931005597622 2.4974671716530272

=== one_c === (12611, 7)  products=10  stores=20
controls: 3 ['promo', 'sin_52', 'cos_52']
pairs train/val: (62586, 10) (1596, 10)
own: 9 products estimated
cross: 58 pairs estimated
own  mean/min/max -0.11154378997659774 -1.210240876753667 0.5847329692663207
cross mean/min/max 0.19633877104616912 -1.0863224146668